# Figury do prezentacji

Zapis do `~/presentation_figures/`. W komórce Slajd 1 zmień `FRAG` / `START_S`.

In [ ]:
import sys, json
from pathlib import Path
import numpy as np, soundfile as sf
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy.ndimage import binary_closing, binary_opening

REPO = Path.cwd(); REPO = REPO.parent if REPO.name == "asr" else REPO
sys.path.insert(0, str(REPO))
OUT = Path("~/presentation_figures").expanduser(); OUT.mkdir(parents=True, exist_ok=True)
EVAL = Path("~/datasets/eval/clarin_fragments").expanduser()
plt.rcParams.update({"savefig.dpi": 200, "figure.dpi": 110, "font.size": 12,
                     "savefig.bbox": "tight", "axes.spines.top": False, "axes.spines.right": False})
BLUE, RED, GREY, YEL, GREEN, ORANGE, PURPLE = "#1f77b4", "#d62728", "#555555", "#ffcc00", "#2ca02c", "#ff7f0e", "#7a3fbf"
def save(fig, name):
    p = OUT / name; fig.savefig(p, facecolor="white"); print("wrote", p)
def mono(p):
    x, sr = sf.read(str(p), dtype="float32"); return (x.mean(1) if x.ndim > 1 else x), sr
print("OUT:", OUT)

## Slajd 1 / 4 — waveformy (15 s): miks -> odseparowane strumienie

In [ ]:
SR = 16000
FRAG = "ccfbb9db__seg00"        # ZMIEŃ na wybrany fragment
CFG = "frcrn_vad_strict"
START_S = None                  # None = auto (15-s okno z największym nakładaniem); lub podaj liczbę
DUR_S = 15.0
fd = EVAL / FRAG
mix, _ = mono(fd / f"{FRAG}.wav"); A, _ = mono(fd / "sweep" / CFG / "stream_A.wav"); B, _ = mono(fd / "sweep" / CFG / "stream_B.wav")
ov = [(r["start"], r["end"]) for r in json.load(open(fd / "sweep" / CFG / "routing.json"))["overlap_regions"]]
if START_S is None:
    best, START_S = -1.0, 0.0
    for s0 in np.arange(0, max(len(mix)/SR - DUR_S, 0) + 0.1, 1.0):
        d = sum(min(e, s0+DUR_S) - max(s, s0) for s, e in ov if min(e, s0+DUR_S) > max(s, s0))
        if d > best: best, START_S = d, float(s0)
i0, i1 = int(START_S*SR), int((START_S+DUR_S)*SR); crop = lambda x: x[i0:i1]
tt = lambda x: START_S + np.arange(len(x))/SR
fig, axes = plt.subplots(3, 1, figsize=(11, 5.0), sharex=True)
for ax, (sig, lab, col) in zip(axes, [(crop(mix), "Miks\n(wejście)", GREY), (crop(A), "Mówca A", BLUE), (crop(B), "Mówca B", RED)]):
    ax.plot(tt(sig), sig, lw=0.6, color=col)
    m = max(float(np.abs(sig).max()), 1e-3); ax.set_ylim(-1.05*m, 1.05*m); ax.set_yticks([])
    ax.set_ylabel(lab, rotation=0, ha="right", va="center", labelpad=45)
    for s, e in ov:
        if min(e, START_S+DUR_S) > max(s, START_S): ax.axvspan(max(s, START_S), min(e, START_S+DUR_S), color=YEL, alpha=0.20, lw=0)
axes[-1].set_xlabel("czas [s]"); axes[-1].set_xlim(START_S, START_S+DUR_S)
axes[0].set_title(f"Separacja: jeden zmiksowany sygnał -> dwa strumienie per mówca   (żółte = nakładanie; okno {START_S:.0f}-{START_S+DUR_S:.0f} s)")
fig.tight_layout(); save(fig, "slajd1_waveformy.png"); plt.show()

## Slajd 2 — skład próbki PolSESS (schemat)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5)); ax.axis("off"); ax.set_xlim(-0.3, 10.2); ax.set_ylim(-0.4, 4.4)
layers = [("Mówca 1 (+ pogłos)", BLUE), ("Mówca 2 (+ pogłos)", RED), ("Tło / scena akustyczna", GREEN), ("Zdarzenia dźwiękowe", ORANGE)]
for i, (name, col) in enumerate(layers):
    ax.add_patch(mpatches.FancyBboxPatch((0, i), 4, 0.72, boxstyle="round,pad=0.02", fc=col, ec="none", alpha=0.9))
    ax.text(2, i + 0.36, name, ha="center", va="center", color="white", fontsize=12, fontweight="bold")
yc = (len(layers) - 1) / 2 + 0.36
ax.text(4.45, yc, "+", fontsize=30, va="center", ha="center")
ax.annotate("", xy=(6.2, yc), xytext=(4.9, yc), arrowprops=dict(arrowstyle="-|>", lw=2.5, color="#333"))
ax.add_patch(mpatches.FancyBboxPatch((6.3, yc - 0.45), 3.6, 0.9, boxstyle="round,pad=0.02", fc=GREY, ec="none"))
ax.text(8.1, yc, "Miks (1 próbka)\n4 s · 8 kHz", ha="center", va="center", color="white", fontsize=12, fontweight="bold")
ax.set_title("Skład próbki PolSESS  (augmentacja MM-IPC: zmienna złożoność tła)")
fig.tight_layout(); save(fig, "slajd2_polsess.png"); plt.show()

## Slajd 3 — diagram pipeline'u

In [ ]:
fig, ax = plt.subplots(figsize=(13.5, 4.6)); ax.axis("off"); ax.set_xlim(0, 14.5); ax.set_ylim(0, 5)
ymid, ytop, ybot, h = 2.0, 3.3, 0.5, 0.9
def box(x, y, w, txt, fc):
    ax.add_patch(mpatches.FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.05", fc=fc, ec="black", lw=1.2))
    ax.text(x + w/2, y + h/2, txt, ha="center", va="center", fontsize=10)
    return {"l": x, "r": x+w, "cx": x+w/2, "b": y, "t": y+h, "cy": y+h/2}
def harr(p1, p2): ax.annotate("", xy=p2, xytext=p1, arrowprops=dict(arrowstyle="-|>", lw=1.6, color="#333", connectionstyle="arc3,rad=0"))
def elbow(p1, p2): ax.annotate("", xy=p2, xytext=p1, arrowprops=dict(arrowstyle="-|>", lw=1.6, color="#333", connectionstyle="angle,angleA=0,angleB=90,rad=0"))
IN  = box(0.1, ymid, 1.25, "Wejście\n(miks)", "#eeeeee")
d1  = box(1.65, ymid, 1.5, "1. Diaryzacja\n(pyannote)", "#cfe8ff")
rt  = box(3.45, ymid, 1.3, "2. Routing", "#cfe8ff")
e3a = box(5.2, ytop, 1.95, "3a. Enhancement\n(FRCRN)", "#d9f0d3")
s3b = box(5.2, ybot, 1.95, "3b. Separacja\n(SepFormer)", "#ffe0cc")
p3c = box(7.45, ybot, 1.75, "3c. Post-proc\nVAD + BWE", "#ffe0cc")
asm = box(9.55, ymid, 1.55, "4. Assembly\n(ECAPA)", "#e6d7f2")
asr = box(11.4, ymid, 1.5, "5. ASR\n(WhisperX)", "#fde2e4")
OUT_= box(13.2, ymid, 1.2, "Wyjście\n(transkrypty)", "#eeeeee")
harr((IN["r"], IN["cy"]), (d1["l"], d1["cy"])); harr((d1["r"], d1["cy"]), (rt["l"], rt["cy"]))
elbow((rt["r"], rt["cy"]+0.12), (e3a["cx"], e3a["b"])); elbow((rt["r"], rt["cy"]-0.12), (s3b["cx"], s3b["t"]))
harr((s3b["r"], s3b["cy"]), (p3c["l"], p3c["cy"]))
elbow((e3a["r"], e3a["cy"]), (asm["cx"], asm["t"])); elbow((p3c["r"], p3c["cy"]), (asm["cx"], asm["b"]))
harr((asm["r"], asm["cy"]), (asr["l"], asr["cy"])); harr((asr["r"], asr["cy"]), (OUT_["l"], OUT_["cy"]))
ax.set_title("Pipeline przetwarzania")
fig.tight_layout(); save(fig, "slajd3_pipeline.png"); plt.show()

## Slajd 3.5 — referencyjne kanały + miks pokolorowany wg aktywności

In [ ]:
# Slajd 3.5: referencyjne kanały mówców (oracle debleed) + miks pokolorowany wg AKTYWNOŚCI ENERGETYCZNEJ.
GOT = Path("~/datasets/clarin_gotowy/gotowy").expanduser()
refA, SR = mono(GOT / "debleed" / "442dd69e_L.wav")
refB, _ = mono(GOT / "debleed" / "442dd69e_R.wav")
mix, _ = mono(GOT / "442dd69e.wav")
n = min(len(refA), len(refB), len(mix)); refA, refB, mix = refA[:n], refB[:n], mix[:n]
def energy_mask(x, sr, frame_ms=30, thr_rel=0.07, close_ms=120, open_ms=40):
    win = max(int(sr*frame_ms/1000), 1)
    env = np.sqrt(np.convolve(x.astype(np.float64)**2, np.ones(win)/win, mode="same"))
    m = env > max(thr_rel*float(env.max()), 1e-4)
    m = binary_closing(m, structure=np.ones(max(int(sr*close_ms/1000), 1)))   # zszyj pauzy wewnątrz słów
    m = binary_opening(m, structure=np.ones(max(int(sr*open_ms/1000), 1)))    # usuń mikro-zakłócenia
    return m
mA, mB = energy_mask(refA, SR), energy_mask(refB, SR)
DUR = 15.0; win = int(DUR*SR); both = (mA & mB).astype(np.int64); cum = np.concatenate([[0], np.cumsum(both)])
best, W0 = -1, 0.0
for s in range(0, max(n-win, 0)+1, SR):
    d = cum[s+win] - cum[s]
    if d > best: best, W0 = d, s/SR
i0 = int(W0*SR); i1 = i0+win; W1 = W0+DUR
aA, aB, mc = mA[i0:i1], mB[i0:i1], mix[i0:i1]
cls = np.where(aA & aB, 3, np.where(aA, 1, np.where(aB, 2, 0)))
cmap = {1: BLUE, 2: RED, 3: PURPLE}; t = W0 + np.arange(len(mc))/SR
fig, axes = plt.subplots(3, 1, figsize=(11, 5.6), sharex=True)
for ax, (sig, lab, col) in zip(axes[:2], [(refA[i0:i1], "Mówca A\n(referencja)", BLUE), (refB[i0:i1], "Mówca B\n(referencja)", RED)]):
    ax.plot(t, sig, lw=0.6, color=col)
    m = max(float(np.abs(sig).max()), 1e-3); ax.set_ylim(-1.05*m, 1.05*m); ax.set_yticks([])
    ax.set_ylabel(lab, rotation=0, ha="right", va="center", labelpad=52)
axm = axes[2]; axm.plot(t, mc, lw=0.5, color="#d4d4d4")
k = 0
while k < len(cls):
    c = cls[k]; j = k
    while j < len(cls) and cls[j] == c: j += 1
    if c in cmap: axm.plot(t[k:j], mc[k:j], lw=0.7, color=cmap[c])
    k = j
m = max(float(np.abs(mc).max()), 1e-3); axm.set_ylim(-1.05*m, 1.05*m); axm.set_yticks([])
axm.set_ylabel("Miks\n(kolory=aktywność)", rotation=0, ha="right", va="center", labelpad=52)
axm.set_xlabel("czas [s]"); axm.set_xlim(W0, W1)
fig.suptitle(f"Referencyjne kanały mówców + miks pokolorowany wg aktywności  (442dd69e, okno {W0:.0f}-{W1:.0f} s)", y=0.99)
fig.legend(handles=[mpatches.Patch(color=BLUE, label="mówca A"), mpatches.Patch(color=RED, label="mówca B"), mpatches.Patch(color=PURPLE, label="nakładanie")],
           loc="upper center", ncol=3, bbox_to_anchor=(0.5, 0.945), framealpha=0.9)
fig.subplots_adjust(top=0.86); save(fig, "slajd35_waveformy.png"); plt.show()

## Slajd 5 — wstępne metryki

In [ ]:
# WSTĘPNE liczby (zaktualizuj po anotacji większej liczby nagrań)
groups = ["5 fragmentów\n(gęste nakładanie)", "Nagranie 16 min\n(442dd69e)"]
baseline = [24.7, 15.0]; pipeline = [17.2, 12.3]; x = np.arange(len(groups)); w = 0.36
fig, ax = plt.subplots(figsize=(7.6, 4.6))
ax.bar(x - w/2, baseline, w, label="bez pipeline'u (sam ASR na miksturze)", color=GREY)
ax.bar(x + w/2, pipeline, w, label="pipeline (FRCRN + separacja)", color=BLUE)
for i, (b, p) in enumerate(zip(baseline, pipeline)):
    ax.text(i - w/2, b + 0.3, f"{b:.1f}", ha="center", fontsize=10); ax.text(i + w/2, p + 0.3, f"{p:.1f}", ha="center", fontsize=10)
ax.set_xticks(x); ax.set_xticklabels(groups); ax.set_ylabel("WER [%]  (mniej = lepiej)"); ax.set_ylim(0, max(baseline)*1.18)
ax.set_title("Wstępne wyniki: WER pipeline vs sam ASR na miksturze  (n = 5 + 1)")
ax.legend(); fig.tight_layout(); save(fig, "slajd5_metryki.png"); plt.show()